In [1]:
import pandas as pd
import requests
import json
from utlis import get_api_entry_by_llm, get_data, merge_by_qid
from question_classify import classify

In [ ]:
classified_val_data = classify("data/val.json","LLM small")

KeyError: 'choices'

In [2]:
classified_test_data = classify("data/test.json","LLM small")


In [4]:
classified_test_1_data = classify("data/test_1.json","LLM small")

In [ ]:
with open("data/classified_val.json", "w", encoding="utf-8") as f:
    json.dump(classified_val_data, f, ensure_ascii=False, indent=4)
with open("data/classified_test.json", "w", encoding="utf-8") as f:
    json.dump(classified_test_data, f, ensure_ascii=False, indent=4)



In [7]:
with open("data/classified_test_1.json", "w", encoding="utf-8") as f:
    json.dump(classified_test_1_data, f, ensure_ascii=False, indent=4)

# STEM

In [12]:
with open('data/classified_val.json', 'r',encoding="utf-8") as file:
    classified_val_data = json.load(file)

JSONDecodeError: Expecting value: line 1 column 1 (char 0)

In [13]:
stem_data = []
for data in classified_val_data:
    if data['label']=="STEM":
        stem_data.append(data)

NameError: name 'classified_val_data' is not defined

In [4]:
for data in stem_data:
    print(f"{data['qid']}. {data['answer']}\n")

val_0004. B

val_0005. C

val_0012. B

val_0013. D

val_0016. B

val_0020. C

val_0023. A

val_0024. A

val_0025. B

val_0026. B

val_0027. C

val_0033. C

val_0038. B

val_0040. E

val_0042. B

val_0043. D

val_0047. A

val_0049. B

val_0051. A

val_0059. A

val_0064. A

val_0065. B

val_0066. E

val_0069. A

val_0071. C

val_0076. D

val_0078. A

val_0082. B

val_0084. D

val_0085. A

val_0087. B

val_0090. B

val_0091. A

val_0093. D



In [ ]:
import json
import re
from typing import List, Dict, Any

def extract_answers(raw_output: str) -> List[Dict[str, Any]]:
    """
    Trích JSON array dạng:
    [
      {"qid": "val_0001", "answer": "A"},
      {"qid": "val_0002", "answer": "C"},
      ...
    ]
    từ output của mô hình.

    Trả về: list[{"qid": str, "answer": str}]
    """

    raw_output = raw_output.strip()

    # 1. Try parse toàn bộ chuỗi như JSON trước (trường hợp model ngoan)
    try:
        data = json.loads(raw_output)
        if isinstance(data, list):
            # optional: validate phần tử
            cleaned = []
            for item in data:
                if (
                    isinstance(item, dict)
                    and "qid" in item
                    and "answer" in item
                ):
                    cleaned.append({"qid": str(item["qid"]), "answer": str(item["answer"]).strip()})
            if cleaned:
                return cleaned
    except json.JSONDecodeError:
        pass  # fallback to regex

    # 2. Fallback: tìm JSON array trong text (trường hợp có Phase 1 in ra cùng)
    # Tìm đoạn '[' ... ']' cuối cùng trông giống JSON array
    # Regex này "tham" nhưng hữu dụng trong thực tế.
    array_pattern = re.compile(r"\[\s*\{.*\}\s*\]", re.DOTALL)
    matches = array_pattern.findall(raw_output)
    if not matches:
        raise ValueError("Không tìm thấy JSON array trong output mô hình.")

    last_array_str = matches[-1].strip()

    try:
        data = json.loads(last_array_str)
    except json.JSONDecodeError as e:
        raise ValueError(f"JSON array cuối cùng parse không được: {e}") from e

    if not isinstance(data, list):
        raise ValueError("Đoạn parse được không phải là JSON array.")

    result = []
    for item in data:
        if not isinstance(item, dict):
            continue
        qid = str(item.get("qid", "")).strip()
        ans = str(item.get("answer", "")).strip()
        if not qid or not ans:
            continue
        result.append({"qid": qid, "answer": ans})

    if not result:
        raise ValueError("JSON array không chứa phần tử hợp lệ (thiếu qid/answer).")

    return result


In [9]:
system_prompt = '''
PHASE 1 — EXTERNAL REASONING (HIỂN THỊ BƯỚC GIẢI)
====================================================
Bạn là mô hình chuyên gia giải các bài toán STEM (Toán, Lý, Hóa, Sinh, Thống kê, Công nghệ, Kinh tế học kỹ thuật).

Đối với MỖI câu hỏi trong danh sách đầu vào, bạn phải:
1. Đọc nội dung câu hỏi.
2. Đọc danh sách choices (mảng không có A/B/C/D).
3. Gán nhãn vị trí cho từng lựa chọn:
      choice[0] → A
      choice[1] → B
      choice[2] → C
      choice[3] → D
      ...
4. Giải bài toán theo trình tự rõ ràng:
   (a) Xác định dữ kiện và yêu cầu cần tìm.
   (b) Gọi tên công thức hoặc định luật phù hợp.
   (c) Thay số, biến đổi, rút gọn, kiểm tra sai số.
   (d) Tính ra kết quả cuối cùng (dạng số hoặc biểu thức).
   (e) So sánh kết quả thu được với từng lựa chọn.
   (f) Xác định lựa chọn đúng theo vị trí (A/B/C/D/...).

Bạn được phép:
- Hiển thị toàn bộ chain-of-thought, tính toán, lập luận, công thức.
- Dùng LaTeX để biểu diễn công thức.

KHÔNG ĐƯỢC:
- Nhảy thẳng tới đáp án mà không giải thích.
- Bỏ qua bước so sánh với từng lựa chọn.

SAU KHI HOÀN THÀNH PHẦN GIẢI CỦA TẤT CẢ CÂU HỎI,
bạn phải CHUYỂN sang PHASE 2 và CHỈ TRẢ VỀ JSON ARRAY DUY NHẤT theo format yêu cầu.

====================================================
PHASE 2 — FINAL OUTPUT (CHỈ JSON ARRAY)
=========================================
Trong phase này, bạn phải TRẢ VỀ DUY NHẤT một JSON array.

Mỗi phần tử phải có dạng:
{
  "qid": "<qid>",
  "answer": "<A|B|C|D|E|...>"
}

YÊU CẦU BẮT BUỘC:
- KHÔNG được ghi bất kỳ văn bản, nhãn phase, giải thích hay ký tự nào trước hoặc sau JSON array.
- Chỉ được xuất đúng một JSON array chứa đúng số lượng câu hỏi trong user prompt.
- answer phải là A/B/C/D/E/... dựa theo **vị trí** của lựa chọn đúng.
- KHÔNG được in lại nội dung đáp án, chỉ in chữ cái.
- KHÔNG được in chain-of-thought trong Phase 2.
- KHÔNG được in văn bản ngoài JSON (nếu có → sai format).

Ví dụ hợp lệ:
[
  {"qid": "q1", "answer": "C"},
  {"qid": "q2", "answer": "A"},
  {"qid": "q3", "answer": "D"}
]
'''


In [10]:
vnpt_model_large = get_api_entry_by_llm("LLM large")

In [11]:
results = []
for i in range(0, len(stem_data), 5):
    question_str = ""
    if i + 5>len(stem_data):
        question_str = "\n\n".join([f"{stem_data[j]['qid']}. {stem_data[j]['question']} \n {stem_data[j]['choices']}" for j in range(i, len(stem_data))]) 
    else:
        question_str = "\n\n".join([f"{stem_data[i+j]['qid']}. {stem_data[i+j]['question']} \n {stem_data[i+j]['choices']}" for j in range(5)])
    user_prompt = f'''
        Danh sách các câu hỏi:
        {question_str}
        '''
    headers = { 
            'Authorization': vnpt_model_large["authorization"], 
            'Token-id': vnpt_model_large["tokenId"], 
            'Token-key': vnpt_model_large["tokenKey"], 
            'Content-Type': 'application/json', 
        }
    json_data = {
            'model': 'vnptai_hackathon_large', 
            'messages': [
                {'role': 'system', 'content': system_prompt},
                {'role': 'user', 'content': user_prompt},
            ],
            'temperature': 0.0, 
            'top_p': 1.0, 
            'top_k': 0, 
            'n': 1, 
            'max_completion_tokens': 2048,
            'seed': 1
        }
    endpoint = "/v1/chat/completions/vnptai-hackathon-large"
    response = requests.post(f'https://api.idg.vnpt.vn/data-service{endpoint}', headers=headers, json=json_data) 
    response_js = response.json()
    print(response_js["choices"][0]["message"]["content"])
    result = extract_answers(response_js["choices"][0]["message"]["content"])
    results.extend(result)
    

NameError: name 'stem_data' is not defined

In [14]:
results

[{'qid': 'val_0004', 'answer': 'B'},
 {'qid': 'val_0005', 'answer': 'C'},
 {'qid': 'val_0012', 'answer': 'B'},
 {'qid': 'val_0013', 'answer': 'D'},
 {'qid': 'val_0016', 'answer': 'B'},
 {'qid': 'val_0023', 'answer': 'A'},
 {'qid': 'val_0024', 'answer': 'A'},
 {'qid': 'val_0026', 'answer': 'B'},
 {'qid': 'val_0027', 'answer': 'C'},
 {'qid': 'val_0033', 'answer': 'C'},
 {'qid': 'val_0038', 'answer': 'B'},
 {'qid': 'val_0040', 'answer': 'D'},
 {'qid': 'val_0042', 'answer': 'B'},
 {'qid': 'val_0043', 'answer': 'D'},
 {'qid': 'val_0047', 'answer': 'A'},
 {'qid': 'val_0049', 'answer': 'B'},
 {'qid': 'val_0051', 'answer': 'A'},
 {'qid': 'val_0059', 'answer': 'A'},
 {'qid': 'val_0064', 'answer': 'A'},
 {'qid': 'val_0065', 'answer': 'B'},
 {'qid': 'val_0066', 'answer': 'E'},
 {'qid': 'val_0069', 'answer': 'A'},
 {'qid': 'val_0071', 'answer': 'A'},
 {'qid': 'val_0076', 'answer': 'D'},
 {'qid': 'val_0078', 'answer': 'A'},
 {'qid': 'val_0082', 'answer': 'B'},
 {'qid': 'val_0084', 'answer': 'D'},
 